# CA-APSRG Full Project Runner di Google Colab + Streamlit Ngrok

Notebook ini mengubah workflow lokal project menjadi workflow Colab lengkap.

Isi notebook:

1. Clone project dari GitHub atau pakai project yang sudah ada di `/content`.
2. Install dependency project dan `pyngrok`.
3. Menyiapkan dataset ke path yang sama dengan config lokal: `data/raw/Retinal Vessel`.
4. Convert dataset ke PNG working folder: `data/working_png`.
5. Build manifest: `data/manifests/manifest.csv`.
6. Run experiment 1-6 memakai `scripts/05_run_all_experiments.py`.
7. Menjelaskan/mengecek lokasi output eksperimen.
8. Menjalankan Streamlit `app.py` di Colab dan membuka akses publik via ngrok.

## Yang Harus Dipersiapkan

- Repo GitHub berisi project ini: `app.py`, `requirements.txt`, `configs/`, `src/`, `scripts/`, dan notebook ini.
- Dataset retina asli di Google Drive atau ZIP upload. Struktur yang disarankan setelah disalin ke project:

```text
data/raw/Retinal Vessel/
  DRIVE/
  STARE/
  CHASEDB1/
```

- Akun ngrok dan authtoken untuk membuat public URL Streamlit.

Catatan: menjalankan eksperimen 1-6 penuh di Colab CPU bisa lama dan menghasilkan output besar. Untuk demo cepat, jalankan 1 eksperimen dulu dengan `EXPERIMENTS_TO_RUN = "exp3_balanced_main"`.

## 1. Konfigurasi Colab

Ubah nilai di cell ini sesuai kebutuhan. Untuk menjalankan semua eksperimen, gunakan `EXPERIMENTS_TO_RUN = "all"`.

In [ ]:
from pathlib import Path

# Repository project
REPO_URL = "https://github.com/nicolauseuclides512/ca_apsrg_retinal_project.git"  # @param {type:"string"}
BRANCH = ""  # @param {type:"string"}
PROJECT_DIR = Path("/content/ca_apsrg_retinal_project")
FORCE_RECLONE = False  # @param {type:"boolean"}

# Dataset source: "drive", "zip", "already_in_repo", atau "skip"
DATA_SOURCE = "drive"  # @param ["drive", "zip", "already_in_repo", "skip"]
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Retinal Vessel"  # @param {type:"string"}
FORCE_REFRESH_RAW_DATA = False  # @param {type:"boolean"}

# Data preparation
RUN_DATA_CONVERSION = True  # @param {type:"boolean"}
RUN_MANIFEST_BUILD = True  # @param {type:"boolean"}

# Experiment runner
RUN_EXPERIMENTS = True  # @param {type:"boolean"}
EXPERIMENTS_TO_RUN = "all"  # @param {type:"string"}
# contoh EXPERIMENTS_TO_RUN: "all" atau "exp3_balanced_main" atau "exp1_recall_oriented exp2_precision_oriented"
CLEAN_OUTPUTS_BEFORE_RUN = True  # @param {type:"boolean"}
METRICS_ONLY_ZIP = False  # @param {type:"boolean"}

# Streamlit + ngrok
RUN_STREAMLIT_WITH_NGROK = True  # @param {type:"boolean"}
STREAMLIT_PORT = 8501  # @param {type:"integer"}
NGROK_AUTHTOKEN = ""  # @param {type:"string"}

# Optional backup output ke Google Drive
COPY_OUTPUTS_TO_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ca_apsrg_colab_outputs"  # @param {type:"string"}

PROJECT_RAW_ROOT = PROJECT_DIR / "data" / "raw" / "Retinal Vessel"
WORKING_PNG_ROOT = PROJECT_DIR / "data" / "working_png"
MANIFEST_PATH = PROJECT_DIR / "data" / "manifests" / "manifest.csv"

print("Project dir:", PROJECT_DIR)
print("Raw dataset target:", PROJECT_RAW_ROOT)
print("Working PNG target:", WORKING_PNG_ROOT)
print("Manifest target:", MANIFEST_PATH)

## 2. Clone Repo dan Install Dependency

Cell ini akan clone repo, masuk ke project root, lalu install dependency dari `requirements.txt` dan `pyngrok`.

In [ ]:
import os
import shutil
import subprocess
import sys

def run_cmd(command: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    cwd = cwd or PROJECT_DIR
    print("\n$", " ".join(str(item) for item in command))
    return subprocess.run(command, cwd=cwd, check=check)

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    clone_cmd = ["git", "clone", "--depth", "1"]
    if BRANCH.strip():
        clone_cmd.extend(["--branch", BRANCH.strip()])
    clone_cmd.extend([REPO_URL, str(PROJECT_DIR)])
    run_cmd(clone_cmd, cwd=Path("/content"))
else:
    print("Project folder already exists:", PROJECT_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pyngrok"])

print("\nSetup done")
print("cwd:", Path.cwd())

## 3. Siapkan Dataset ke `data/raw/Retinal Vessel`

Pilih salah satu mode:

- `drive`: mount Google Drive lalu copy folder dataset dari `DRIVE_DATASET_DIR`.
- `zip`: upload ZIP dataset, lalu notebook mencoba menemukan folder yang berisi DRIVE/STARE/CHASEDB1.
- `already_in_repo`: dataset sudah berada di repo pada `data/raw/Retinal Vessel`.
- `skip`: lewati setup dataset, berguna kalau hanya ingin membuka Streamlit dari output yang sudah ada.

In [ ]:
import zipfile

DATASET_NAMES = {"DRIVE", "STARE", "CHASEDB1"}

def looks_like_dataset_root(path: Path) -> bool:
    if not path.is_dir():
        return False
    child_names = {p.name.upper() for p in path.iterdir() if p.is_dir()}
    return bool(DATASET_NAMES & child_names)

def find_dataset_root(base_dir: Path) -> Path:
    if looks_like_dataset_root(base_dir):
        return base_dir
    matches = []
    for path in base_dir.rglob("*"):
        if path.is_dir() and looks_like_dataset_root(path):
            matches.append(path)
    if not matches:
        raise FileNotFoundError(
            "Tidak menemukan folder dataset yang berisi DRIVE/STARE/CHASEDB1. "
            f"Cek isi: {base_dir}"
        )
    matches = sorted(matches, key=lambda p: len(p.parts))
    return matches[0]

def copy_dataset_to_project(source_root: Path, target_root: Path, force: bool = False) -> None:
    if target_root.exists() and force:
        shutil.rmtree(target_root)
    target_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_root, target_root, dirs_exist_ok=True)
    print("Dataset copied")
    print("from:", source_root)
    print("to  :", target_root)

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    source_root = find_dataset_root(Path(DRIVE_DATASET_DIR))
    copy_dataset_to_project(source_root, PROJECT_RAW_ROOT, FORCE_REFRESH_RAW_DATA)
elif DATA_SOURCE == "zip":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Tidak ada ZIP yang diupload")
    zip_name = next(iter(uploaded))
    extract_dir = Path("/content/dataset_upload")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(extract_dir)
    source_root = find_dataset_root(extract_dir)
    copy_dataset_to_project(source_root, PROJECT_RAW_ROOT, FORCE_REFRESH_RAW_DATA)
elif DATA_SOURCE == "already_in_repo":
    if not PROJECT_RAW_ROOT.exists():
        raise FileNotFoundError(f"Dataset belum ada di {PROJECT_RAW_ROOT}")
    print("Using dataset already in repo:", PROJECT_RAW_ROOT)
elif DATA_SOURCE == "skip":
    print("Skip dataset setup")
else:
    raise ValueError(f"Unsupported DATA_SOURCE: {DATA_SOURCE}")

if PROJECT_RAW_ROOT.exists():
    for path in sorted(PROJECT_RAW_ROOT.iterdir()):
        if path.is_dir():
            n_files = sum(1 for _ in path.rglob("*") if _.is_file())
            print(f"- {path.name}: {n_files} files")

## 4. Convert Dataset ke PNG dan Build Manifest

Ini menyamakan workflow lokal:

```bash
python scripts/00_convert_all_to_png.py --overwrite
python scripts/01_build_manifest.py
```

Output penting:

- `data/working_png/`
- `data/manifests/manifest.csv`
- `outputs/reports/png_conversion_report.csv`
- `outputs/reports/manifest_summary.csv`
- `outputs/reports/manifest_validation.csv`

In [ ]:
if RUN_DATA_CONVERSION:
    if not PROJECT_RAW_ROOT.exists():
        raise FileNotFoundError(f"Raw dataset belum tersedia: {PROJECT_RAW_ROOT}")
    run_cmd([sys.executable, "scripts/00_convert_all_to_png.py", "--overwrite"])
else:
    print("Skip dataset conversion")

if RUN_MANIFEST_BUILD:
    if not WORKING_PNG_ROOT.exists():
        raise FileNotFoundError(f"Working PNG root belum tersedia: {WORKING_PNG_ROOT}")
    run_cmd([sys.executable, "scripts/01_build_manifest.py"])
else:
    print("Skip manifest build")

print("\nData preparation status")
print("working_png exists:", WORKING_PNG_ROOT.exists(), WORKING_PNG_ROOT)
print("manifest exists   :", MANIFEST_PATH.exists(), MANIFEST_PATH)
if MANIFEST_PATH.exists():
    import pandas as pd
    manifest_df = pd.read_csv(MANIFEST_PATH)
    display(manifest_df.groupby("dataset", dropna=False).size().reset_index(name="n_images"))
    display(manifest_df.head())

## 5. Jalankan Experiment 1-6

Notebook memakai script project yang sama dengan lokal:

```bash
python scripts/05_run_all_experiments.py --clean
```

Script ini menghasilkan config experiment dan output berikut:

```text
configs/experiments/exp1_recall_oriented.yaml
configs/experiments/exp2_precision_oriented.yaml
configs/experiments/exp3_balanced_main.yaml
configs/experiments/exp4_static_morphology.yaml
configs/experiments/exp5_no_skeleton_guard.yaml
configs/experiments/exp6_no_small_component.yaml

outputs/experiments_exp1_recall_oriented/
outputs/analysis_exp1_recall_oriented/
...
outputs/experiments_exp6_no_small_component/
outputs/analysis_exp6_no_small_component/

outputs/experiments/  # copy Experiment 3 untuk Streamlit default
outputs/analysis/     # copy Experiment 3 untuk Streamlit default
outputs/ca_apsrg_experiments_1_to_6.zip
```

Tips: untuk tes cepat, set `EXPERIMENTS_TO_RUN = "exp3_balanced_main"`.

In [ ]:
def parse_experiments_to_run(value: str) -> list[str]:
    value = value.strip()
    if not value or value.lower() == "all":
        return []
    return [item.strip() for item in value.replace(",", " ").split() if item.strip()]

if RUN_EXPERIMENTS:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Manifest belum tersedia: {MANIFEST_PATH}")
    cmd = [sys.executable, "scripts/05_run_all_experiments.py", "--manifest", "data/manifests/manifest.csv"]
    if CLEAN_OUTPUTS_BEFORE_RUN:
        cmd.append("--clean")
    selected_experiments = parse_experiments_to_run(EXPERIMENTS_TO_RUN)
    if selected_experiments:
        cmd.append("--only")
        cmd.extend(selected_experiments)
    if METRICS_ONLY_ZIP:
        cmd.append("--metrics-only-zip")
    run_cmd(cmd)
else:
    print("Skip experiment runner")

## 6. Cek Lokasi Output dan Ringkasan Hasil

In [ ]:
import pandas as pd

def folder_size_mb(path: Path) -> float:
    if not path.exists():
        return 0.0
    total = sum(file.stat().st_size for file in path.rglob("*") if file.is_file())
    return round(total / (1024 * 1024), 2)

output_roots = [
    PROJECT_DIR / "outputs" / "analysis",
    PROJECT_DIR / "outputs" / "experiments",
    *sorted((PROJECT_DIR / "outputs").glob("analysis_exp*")),
    *sorted((PROJECT_DIR / "outputs").glob("experiments_exp*")),
]
output_status = []
for path in output_roots:
    output_status.append({
        "path": str(path.relative_to(PROJECT_DIR)),
        "exists": path.exists(),
        "size_mb": folder_size_mb(path),
    })
display(pd.DataFrame(output_status))

summary_path = PROJECT_DIR / "outputs" / "experiments" / "metrics_summary.csv"
article_path = PROJECT_DIR / "outputs" / "analysis" / "article_table_mean_std.csv"
improvement_path = PROJECT_DIR / "outputs" / "analysis" / "improvement_by_dataset.csv"

if article_path.is_file():
    print("Article table:")
    display(pd.read_csv(article_path))
if improvement_path.is_file():
    print("Improvement by dataset:")
    display(pd.read_csv(improvement_path)[["dataset", "n_images", "delta_precision_mean", "delta_recall_mean", "delta_f1_score_mean", "delta_iou_mean"]])
if summary_path.is_file():
    print("Metrics summary:")
    df = pd.read_csv(summary_path)
    cols = [c for c in ["dataset", "method", "n_images", "precision_mean", "recall_mean", "f1_score_mean", "iou_mean", "accuracy_mean"] if c in df.columns]
    display(df[cols])

## 7. Perbandingan Experiment 1-6 di Colab

In [ ]:
import matplotlib.pyplot as plt

EXPERIMENT_LABELS = {
    "exp1_recall_oriented": "Experiment 1 - Recall-oriented CA-APSRG",
    "exp2_precision_oriented": "Experiment 2 - Precision-oriented CA-APSRG",
    "exp3_balanced_main": "Experiment 3 - Balanced CA-APSRG Main Result",
    "exp4_static_morphology": "Experiment 4 - Static Morphology Ablation",
    "exp5_no_skeleton_guard": "Experiment 5 - No Skeleton Guard Ablation",
    "exp6_no_small_component": "Experiment 6 - No Small Component Removal Ablation",
}

def weighted_mean(df: pd.DataFrame, column: str) -> float | None:
    if df.empty or column not in df.columns:
        return None
    values = pd.to_numeric(df[column], errors="coerce")
    if "n_images" not in df.columns:
        return float(values.mean())
    weights = pd.to_numeric(df["n_images"], errors="coerce").fillna(0)
    valid = values.notna() & (weights > 0)
    if not valid.any():
        return float(values.mean())
    return float((values[valid] * weights[valid]).sum() / weights[valid].sum())

comparison_rows = []
for key, label in EXPERIMENT_LABELS.items():
    analysis_dir = PROJECT_DIR / "outputs" / f"analysis_{key}"
    experiments_dir = PROJECT_DIR / "outputs" / f"experiments_{key}"
    improvement_csv = analysis_dir / "improvement_by_dataset.csv"
    summary_csv = experiments_dir / "metrics_summary.csv"
    if not improvement_csv.is_file() and not summary_csv.is_file():
        continue
    row = {"experiment_key": key, "experiment": label}
    if improvement_csv.is_file():
        imp = pd.read_csv(improvement_csv)
        row["n_images"] = int(pd.to_numeric(imp.get("n_images", []), errors="coerce").sum())
        for metric in ["precision", "recall", "f1_score", "iou", "accuracy"]:
            row[f"delta_{metric}"] = weighted_mean(imp, f"delta_{metric}_mean")
    if summary_csv.is_file():
        summary = pd.read_csv(summary_csv)
        method_text = summary["method"].astype(str).str.lower()
        ca_df = summary[method_text.str.contains("ca", na=False)]
        baseline_df = summary[(method_text == "apsrg") | method_text.str.contains("baseline", na=False)]
        for metric in ["precision", "recall", "f1_score", "iou", "accuracy"]:
            row[f"ca_{metric}"] = weighted_mean(ca_df, f"{metric}_mean")
            row[f"baseline_{metric}"] = weighted_mean(baseline_df, f"{metric}_mean")
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

for metric in ["delta_precision", "delta_recall", "delta_f1_score", "delta_iou"]:
    if metric not in comparison_df.columns:
        continue
    ax = comparison_df.set_index("experiment")[[metric]].plot(kind="bar", figsize=(10, 4), legend=False)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(metric.replace("_", " ").title())
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 8. Jalankan Streamlit di Colab dengan Ngrok

Sebelum menjalankan cell ini, siapkan ngrok authtoken dari dashboard ngrok. Jangan commit token ke repository.

Cell ini menjalankan perintah setara lokal:

```bash
streamlit run app.py
```

Lalu membuka public URL via ngrok.

In [ ]:
import getpass
import time
from pyngrok import ngrok

if RUN_STREAMLIT_WITH_NGROK:
    token = NGROK_AUTHTOKEN.strip()
    if not token:
        token = getpass.getpass("Paste ngrok authtoken: ")
    ngrok.set_auth_token(token)

    # Bersihkan tunnel lama jika cell dijalankan ulang.
    try:
        ngrok.kill()
    except Exception:
        pass

    try:
        STREAMLIT_PROCESS.terminate()
    except Exception:
        pass

    log_path = Path("/content/ca_apsrg_streamlit.log")
    log_file = log_path.open("w", encoding="utf-8")
    STREAMLIT_PROCESS = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "app.py",
            "--server.headless",
            "true",
            "--server.address",
            "0.0.0.0",
            "--server.port",
            str(STREAMLIT_PORT),
            "--browser.gatherUsageStats",
            "false",
        ],
        cwd=PROJECT_DIR,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)
    tunnel = ngrok.connect(addr=STREAMLIT_PORT, proto="http")
    print("Streamlit public URL:", tunnel.public_url)
    print("Local Colab port:", STREAMLIT_PORT)
    print("Log file:", log_path)
else:
    print("RUN_STREAMLIT_WITH_NGROK=False, skip Streamlit ngrok")

## 9. Stop Streamlit dan Ngrok

Jalankan cell ini saat selesai memakai app agar tunnel ditutup.

In [ ]:
try:
    STREAMLIT_PROCESS.terminate()
    print("Streamlit process terminated")
except Exception as exc:
    print("No Streamlit process to terminate:", exc)

try:
    ngrok.kill()
    print("ngrok tunnels closed")
except Exception as exc:
    print("No ngrok tunnel to close:", exc)

## 10. Optional: Copy Output ke Google Drive

Gunakan ini kalau hasil eksperimen dari Colab ingin disimpan permanen ke Drive.

In [ ]:
if COPY_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_output = Path(DRIVE_OUTPUT_DIR)
    drive_output.parent.mkdir(parents=True, exist_ok=True)
    if drive_output.exists():
        shutil.rmtree(drive_output)
    shutil.copytree(PROJECT_DIR / "outputs", drive_output)
    print("Outputs copied to:", drive_output)
else:
    print("COPY_OUTPUTS_TO_DRIVE=False, skip backup")